In [ ]:
df = (spark.read
      .format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load("/Volumes/workspace/default/data_assignment2/coffee_sales.csv"))

display(df.limit(10))




date,datetime,cash_type,card,money,coffee_name
2024-03-01,2024-03-01T10:15:50.520Z,card,ANON-0000-0000-0001,38.7,Latte
2024-03-01,2024-03-01T12:19:22.539Z,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2024-03-01,2024-03-01T12:20:18.089Z,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2024-03-01,2024-03-01T13:46:33.006Z,card,ANON-0000-0000-0003,28.9,Americano
2024-03-01,2024-03-01T13:48:14.626Z,card,ANON-0000-0000-0004,38.7,Latte
2024-03-01,2024-03-01T15:39:47.726Z,card,ANON-0000-0000-0005,33.8,Americano with Milk
2024-03-01,2024-03-01T16:19:02.756Z,card,ANON-0000-0000-0006,38.7,Hot Chocolate
2024-03-01,2024-03-01T18:39:03.580Z,card,ANON-0000-0000-0007,33.8,Americano with Milk
2024-03-01,2024-03-01T19:22:01.762Z,card,ANON-0000-0000-0008,38.7,Cocoa
2024-03-01,2024-03-01T19:23:15.887Z,card,ANON-0000-0000-0008,33.8,Americano with Milk


In [ ]:
# Write a query to find the top 5 best-selling coffee types by revenue.
top_coffee_types = (df
    .withColumn("revenue", df["money"])
    .groupBy("coffee_name")
    .agg({"revenue": "sum"})
    .withColumnRenamed("sum(revenue)", "total_revenue")
    .orderBy("total_revenue", ascending=False)
    .limit(5)
)

display(top_coffee_types)

coffee_name,total_revenue
Latte,27866.299999999457
Americano with Milk,25269.120000000225
Cappuccino,18034.139999999945
Americano,15062.259999999818
Hot Chocolate,10172.460000000045


In [ ]:
# Write a query to find total revenue by payment type (cash_type).
total_revenue_by_cash_type = (df
    .withColumn("revenue", df["money"])
    .groupBy("cash_type")
    .agg({"revenue": "sum"})
    .withColumnRenamed("sum(revenue)", "total_revenue")
)

display(total_revenue_by_cash_type)

cash_type,total_revenue
card,112245.57999999814
cash,3186.0


In [ ]:
# Write a query using a CTE to find total revenue per month.

df.createOrReplaceTempView("coffee_sales")

query = """
WITH monthly_revenue AS (
  SELECT
    YEAR(date) AS year,
    MONTH(date) AS month,
    SUM(money) AS total_revenue
  FROM coffee_sales
  GROUP BY YEAR(date), MONTH(date)
)
SELECT * FROM monthly_revenue
ORDER BY year, month
"""

monthly_revenue_df = spark.sql(query)
display(monthly_revenue_df.sort("year", "month"))

year,month,total_revenue
2024,3,7050.199999999987
2024,4,6720.560000000001
2024,5,9063.420000000002
2024,6,7758.760000000005
2024,7,6915.940000000006
2024,8,7613.840000000015
2024,9,9988.640000000007
2024,10,13891.16000000004
2024,11,8590.540000000023
2024,12,8237.740000000018


In [ ]:
from pyspark.sql.functions import desc, count

#  Write a query to find the day with the highest number of transactions.
highest_transaction_day = (df
    .groupBy("date")
    .agg(count("*").alias("total_transactions"))
    .orderBy(desc("total_transactions"))
    .limit(1)
)

display(highest_transaction_day)



date,total_transactions
2024-10-11,26


In [ ]:
# Write a query using a window function to rank coffee types by revenue within each month.

from pyspark.sql.functions import rank
from pyspark.sql.window import Window

window = Window.orderBy(desc("total_revenue"))
ranked_coffee_types = (top_coffee_types
    .withColumn("rank", rank().over(window))
)

display(ranked_coffee_types)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


coffee_name,total_revenue,rank
Latte,27866.299999999457,1
Americano with Milk,25269.120000000225,2
Cappuccino,18034.139999999945,3
Americano,15062.259999999818,4
Hot Chocolate,10172.460000000045,5
